# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [ ]:
from pathlib import Path
import time

# ── Chemins ─────────────────────────────────────────────────────────────────
DATA_DIR           = Path("../data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez le Jour 1"

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Delta Lake requiert le package delta-spark
# Vérifiez que votre environnement conda l'inclut avant de démarrer.
spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")


In [ ]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()


---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [ ]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


In [ ]:
# Première requête : distribution des statuts par arrondissement :
# Nom de 'snapshots', taux d'occuoation moyen, écarrt-type de l'occupation
spark.sql("""
    SELECT
        code_arr,
        statut,
        COUNT(*)                       AS nb_snapshots,
        ROUND(AVG(taux_occupation), 4) AS taux_moyen,
        ROUND(STDDEV(taux_occupation), 4) AS taux_ecart_type
    FROM disponibilite
    GROUP BY code_arr, statut
    ORDER BY code_arr, statut
""").show(40)

In [ ]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapsohts, nombre de stations par heure
spark.sql("""
    SELECT
        heure,
        ROUND(AVG(taux_occupation), 4) AS taux_moyen,
        COUNT(*)                       AS nb_snapshots,
        COUNT(DISTINCT station_id)     AS nb_stations
    FROM disponibilite
    GROUP BY heure
    ORDER BY heure
""").show(24)

---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [ ]:
# Table de jours fériés (2022-2023) -- injectée en broadcast
# Source : légifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2022-01-01",), ("2022-04-18",), ("2022-05-01",), ("2022-05-08",),
    ("2022-05-26",), ("2022-06-06",), ("2022-07-14",), ("2022-08-15",),
    ("2022-11-01",), ("2022-11-11",), ("2022-12-25",),
    ("2023-01-01",), ("2023-04-10",), ("2023-05-01",), ("2023-05-08",),
    ("2023-05-18",), ("2023-05-29",), ("2023-07-14",), ("2023-08-15",),
    ("2023-11-01",), ("2023-11-11",), ("2023-12-25",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2022-2023)")


In [ ]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations trè!s fréquentées,ayant plus de 100 observations (snapshots)
# -> Id de la station, noù de la station, arroondissement, nombre d'observations (snapshots)
df_q1 = spark.sql("""
    SELECT /*+ BROADCAST(f) */
        d.station_id,
        d.nom_station,
        d.code_arr,
        COUNT(*) AS nb_observations,
        SUM(CASE WHEN d.velos_meca + d.velos_elec = 0 THEN 1 ELSE 0 END) AS nb_ruptures,
        ROUND(100.0 * SUM(CASE WHEN d.velos_meca + d.velos_elec = 0 THEN 1 ELSE 0 END)
              / COUNT(*), 2) AS pct_rupture
    FROM disponibilite d
    LEFT JOIN jours_feries f
        ON to_date(d.horodatage) = f.date_ferie
    WHERE d.heure BETWEEN 7 AND 10
      AND d.jour_sem BETWEEN 2 AND 6   -- convention Spark dayofweek : 1=dimanche .. 7=samedi
      AND f.date_ferie IS NULL         -- LEFT JOIN + IS NULL = anti-jointure : exclut les jours fériés
    GROUP BY d.station_id, d.nom_station, d.code_arr
    HAVING COUNT(*) > 100
    ORDER BY pct_rupture DESC
    LIMIT 10
""")

df_q1.show(truncate=False)

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [ ]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
df_q2 = spark.sql("""
    SELECT
        est_pluie,
        COUNT(*)                                        AS nb_snapshots,
        ROUND(AVG(taux_occupation), 4)                   AS taux_moyen,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.25), 4) AS q1,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.5), 4)  AS mediane,
        ROUND(PERCENTILE_APPROX(taux_occupation, 0.75), 4) AS q3
    FROM disponibilite
    GROUP BY est_pluie
    ORDER BY est_pluie
""")
df_q2.show()

In [ ]:
# Effet de la pluie par arrondissement
df_q2_arr = spark.sql("""
    SELECT
        code_arr,
        ROUND(AVG(CASE WHEN est_pluie THEN taux_occupation END), 4)     AS taux_pluie,
        ROUND(AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4) AS taux_sec,
        ROUND(
            AVG(CASE WHEN est_pluie THEN taux_occupation END)
            - AVG(CASE WHEN NOT est_pluie THEN taux_occupation END), 4
        ) AS delta_points
    FROM disponibilite
    GROUP BY code_arr
    ORDER BY code_arr
""")
df_q2_arr.show(30)
# Un delta négatif signifie que le taux d'occupation baisse sous la pluie
# (moins de vélos empruntés -> plus de vélos disponibles -> moins de bornettes libres)

### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [ ]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire — presque vides à certaines heures, saturées à d'autres ?
df_q3 = spark.sql("""
    WITH profil_horaire AS (
        SELECT station_id, nom_station, heure, AVG(taux_occupation) AS taux_moyen_heure
        FROM disponibilite
        GROUP BY station_id, nom_station, heure
    )
    SELECT
        station_id,
        nom_station,
        ROUND(MAX(taux_moyen_heure), 4) AS taux_max,
        ROUND(MIN(taux_moyen_heure), 4) AS taux_min,
        ROUND(MAX(taux_moyen_heure) - MIN(taux_moyen_heure), 4) AS amplitude
    FROM profil_horaire
    GROUP BY station_id, nom_station
    ORDER BY amplitude DESC
    LIMIT 15
""")
df_q3.show(truncate=False)

In [ ]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# Votre requête ici :
spark.sql("""
    WITH agg AS (
        SELECT
            station_id,
            AVG(CASE WHEN NOT est_weekend AND NOT est_pluie THEN taux_occupation END) AS taux_semaine_sec,
            AVG(CASE WHEN est_weekend AND est_pluie THEN taux_occupation END)         AS taux_weekend_pluie
        FROM disponibilite
        GROUP BY station_id
    )
    SELECT
        station_id,
        ROUND(taux_semaine_sec, 4)   AS taux_semaine_sec,
        ROUND(taux_weekend_pluie, 4) AS taux_weekend_pluie,
        ROUND(taux_weekend_pluie / taux_semaine_sec, 4) AS ratio
    FROM agg
    WHERE taux_semaine_sec IS NOT NULL
      AND taux_weekend_pluie IS NOT NULL
      AND taux_semaine_sec > 0
    ORDER BY ratio DESC
    LIMIT 10
""").show()

---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [ ]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.

# Fenêtre partitionnée par station et ordonnée dans le temps : LAG/LEAD ne
# regardent jamais au-delà des frontières d'une autre station.
fenetre_station = Window.partitionBy("station_id").orderBy("horodatage")

df_avec_lag = (
    df
    .withColumn("taux_precedent", F.lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_suivant", F.lead("taux_occupation", 1).over(fenetre_station))
    # rowsBetween(-1, 1) = la ligne courante + 1 ligne avant + 1 ligne après
    # -> une fenêtre glissante de 3 lignes, sans réduire le nombre de lignes
    # du résultat (contrairement à un GROUP BY).
    .withColumn(
        "moyenne_mobile_3",
        F.avg("taux_occupation").over(fenetre_station.rowsBetween(-1, 1))
    )
    .select("station_id", "horodatage", "taux_precedent", "taux_occupation",
            "taux_suivant", "moyenne_mobile_3")
)
df_avec_lag.show(truncate=False)

In [ ]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)

fenetre_heure = Window.partitionBy("heure").orderBy(F.desc("taux_occupation"))

df_rank = (
    df
    .withColumn("rang", F.row_number().over(fenetre_heure))
    .filter(F.col("rang") <= 2)   # les 2 stations les plus occupées, par heure
    .select("heure", "rang", "station_id", "nom_station", "taux_occupation")
    .orderBy("heure", "rang")
)
df_rank.show(48, truncate=False)

In [ ]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

fenetre_cumul = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot précédent (delta instantané)
station_cible = 1042   # à adapter selon vos données

df_delta = (
    df
    .filter(F.col("station_id") == station_cible)
    .withColumn("cumul_moyenne", F.avg("taux_occupation").over(fenetre_cumul))
    .withColumn(
        "delta_precedent",
        F.col("taux_occupation") - F.lag("taux_occupation", 1).over(
            Window.partitionBy("station_id").orderBy("horodatage")
        )
    )
    .select("station_id", "horodatage", "taux_occupation", "delta_precedent", "cumul_moyenne")
    .orderBy("horodatage")
)
df_delta.show(truncate=False)

---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [ ]:
from delta.tables import DeltaTable

# Écriture initiale : toutes les données de 2022
df_2022 = df.filter(F.col("annee") == 2022)

t0 = time.perf_counter()
(
    df_2022
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture 2022 : {time.perf_counter()-t0:.1f} s  --  {df_2022.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")


In [ ]:
# Ajout des données 2023 -- mode "append"
df_2023 = df.filter(F.col("annee") == 2023)

t0 = time.perf_counter()
(
    df_2023
    .write
    .format("delta")
    .mode("append")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout 2023 : {time.perf_counter()-t0:.1f} s  --  {df_2023.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [ ]:
# Lecture de la version 0 (uniquement les données 2022)
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version 0 (2022 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = spark.read.format("delta").load(str(DELTA_DISPONIBLE))
print(f"Version courante (2022+2023) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")

### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [ ]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit, current_timestamp

# On prend 500 snapshots existants et on simule une correction du taux_occupation
df_corrections = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
)

# Quelques nouvelles lignes fictives (snapshots manqués)
df_nouveaux = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(50)
    .withColumn("horodatage",
        F.col("horodatage") + F.expr("INTERVAL 2 YEARS"))
    .withColumn("annee", lit(2024))
    .withColumn("source", lit("nouveau_batch"))
)

df_batch = df_corrections.union(df_nouveaux).drop("source")
print(f"Batch entrant : {df_batch.count()} lignes ({df_corrections.count()} corrections + {df_nouveaux.count()} nouvelles)")


In [ ]:
# MERGE INTO : upsert (update + insert)
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        # Condition de correspondance : même station, même horodatage
        "cible.station_id = source.station_id AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)


In [ ]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Votre code ici :
# Historique des versions : v0 = écriture initiale (2022), v1 = append (2023),
# v2 = MERGE INTO -- donc "avant merge" correspond à la version 1.
df_avant_merge = (
    spark.read.format("delta").option("versionAsOf", 1).load(str(DELTA_DISPONIBLE))
)
df_apres_merge = spark.read.format("delta").load(str(DELTA_DISPONIBLE))

taux_avant = (
    df_avant_merge
    .filter((F.col("annee") == 2022) & (F.col("mois") == 1))
    .agg(F.avg("taux_occupation"))
    .first()[0]
)
taux_apres = (
    df_apres_merge
    .filter((F.col("annee") == 2022) & (F.col("mois") == 1))
    .agg(F.avg("taux_occupation"))
    .first()[0]
)

print(f"Taux moyen janvier 2022 AVANT le merge : {taux_avant:.4f}")
print(f"Taux moyen janvier 2022 APRES le merge : {taux_apres:.4f}")
print(f"Écart introduit par le batch de correction : {taux_apres - taux_avant:+.4f}")
print("\nLe time-travel permet ainsi d'auditer précisément l'effet d'une")
print("opération MERGE, sans avoir eu besoin de sauvegarder une copie manuelle")
print("de la table avant modification -- c'est Delta Lake qui s'en charge.")